# Deep Dive 1 — Snowpark Container Services

**Extends:** [2.5 — Bring your own model](../Domain%202.0%20-%20Gen%20AI%20Functions/2.5.ipynb) · also relevant to [3.3 — Cost and governance](../Domain%203.0%20-%20Gen%20AI%20Governance/3.3.ipynb) · first introduced in [1.1 — Gen AI principles and features](../Domain%201.0%20-%20Gen%20AI%20Overview/1.1.ipynb)

## The problem this solves

Your data scientist has a model that needs a GPU and three Python packages Snowflake does not ship. Or the vendor gave you a Docker image and no source. Or the inference server has to answer an HTTP call in 50 milliseconds, not in a batch overnight. A warehouse cannot run any of that: it runs SQL, not containers.

Snowpark Container Services is the answer to "I need to run *my* container, inside the account, under the same grants as everything else". Most of the difficulty is not conceptual — it is that four objects have to be created in the right order, and the failures when you get it wrong are quiet.

## What you will be able to do

- Build the four SPCS objects in an order that works, and say why each step depends on the one before it
- Write a service specification and know which fields are load-bearing
- Choose between a long-running service and a job service on the shape of the workload
- Reach a container from SQL through a service function, and grant that access to another role
- Find out what SPCS is costing you, and stop it costing that

## Before you start

- `ACCOUNTADMIN` for the account-level grants in the first cell. Everything after that runs as a purpose-built role.
- Docker, and somewhere to build an image. Half of this deep dive happens in a terminal.
- Read [1.1](../Domain%201.0%20-%20Gen%20AI%20Overview/1.1.ipynb) first if "database role" and "account privilege" are not yet distinct in your head — the grants below assume that distinction.

📖 **Snowflake documentation for this notebook**
- [Snowpark Container Services overview](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)
- [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)
- [Working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)
- [Service specification reference](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/specification-reference)
- [CREATE SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-service)
- [Service functions and additional considerations](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)
- [SPCS cost views](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/accounts-orgs-usage-views)

---
## 1. Four objects, and why each exists

Each one is there because of a specific limitation of the one before it.

```
IMAGE REPOSITORY   an OCI registry that lives inside Snowflake
        |          (Snowflake will not pull from your private registry)
COMPUTE POOL       a collection of one or more virtual machine nodes
        |          (containers need machines, and warehouses run SQL, not containers)
SERVICE            containers running on the pool, defined by a specification
        |          (long-running) - or EXECUTE JOB SERVICE (runs once, exits)
SERVICE FUNCTION   a SQL function bound to an endpoint on the service
                   (SQL cannot speak HTTP on its own)
```

**One sentence to keep:** the compute pool is what you pay for, the service is the workload, and the service function is the SQL door into it.

An *image repository* is an OCIv2-compliant registry, created with `CREATE IMAGE REPOSITORY`, that Docker can push to. A *compute pool* is, in the documentation's own words, "a collection of one or more virtual machine (VM) nodes" — it is an account-level object, like a warehouse, and several services can share one.

| Object | Scope | Billed? |
|---|---|---|
| Image repository | schema | storage only |
| Compute pool | **account** | **yes — node-hours** |
| Service | schema | not directly, but it occupies the pool |
| Service function | schema | warehouse credits for the calling query |

That compute pools are account-level, not schema-level, is worth fixing in memory. It is the same shape as a warehouse: created once, granted to roles, shared by workloads.

### Why not a Snowpark-optimized warehouse?

Because it still is not a container. A Snowpark-optimized warehouse is single-node with configurable memory via `RESOURCE_CONSTRAINT` — `MEMORY_1X` (16 GB), `MEMORY_16X` (256 GB, minimum size M) and `MEMORY_64X` (1 TB, minimum size L, in preview on AWS). That solves "my Python ran out of memory". It does **not** give you a GPU and it does not run your image. Only compute pools offer GPUs.

→ [More on compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

---
## 2. The setup sequence

Ten steps. The order is not stylistic — each step consumes something the previous one produced.

| # | Step | Why it has to be here |
|---|---|---|
| 1 | `USE ROLE ACCOUNTADMIN` | only ACCOUNTADMIN can make the account-level grants below |
| 2 | `CREATE ROLE` for the workload | you want the service owned by a purpose-built role, not by ACCOUNTADMIN |
| 3 | `CREATE DATABASE` / `SCHEMA` | the image repository and the service need a home |
| 4 | `CREATE WAREHOUSE` + `GRANT USAGE` | the service function's caller and any `QUERY_WAREHOUSE` need one |
| 5 | `GRANT BIND SERVICE ENDPOINT ON ACCOUNT` | required **before** creating a service with a public endpoint |
| 6 | `CREATE COMPUTE POOL` + `GRANT USAGE, MONITOR` | the service cannot be created without a pool to land on |
| 7 | `GRANT ROLE … TO USER` | so you can switch into the role you just built |
| 8 | `CREATE IMAGE REPOSITORY` as the workload role | you need its URL before you can tag an image |
| 9 | **Build and push the image** | the service references an image that must already exist |
| 10 | `CREATE SERVICE` | last, because it consumes everything above |

### What each reordering actually produces

| Mistake | What you see |
|---|---|
| `CREATE SERVICE` before the image is pushed | the service is created, then containers sit unable to start — an image pull failure, not a SQL error |
| Skipped `BIND SERVICE ENDPOINT` | `CREATE SERVICE` is refused when the specification declares a public endpoint |
| Built the image on Apple Silicon without `--platform linux/amd64` | `docker push` succeeds, containers crash-loop. SPCS requires `linux/amd64` |
| Tagged before creating the repository | the tag carries the wrong registry hostname and the push is rejected |
| `DROP COMPUTE POOL` with a service still on it | refused — all running services must be stopped first |

`CREATE COMPUTE POOL` requires `MIN_NODES`, `MAX_NODES` and `INSTANCE_FAMILY`. Everything else has a default.

→ [More on CREATE SERVICE privileges](https://docs.snowflake.com/en/sql-reference/sql/create-service)

In [ ]:
%%sql
-- Step 1-2: role
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS SPCS_ROLE;

-- Step 3: container objects
CREATE DATABASE IF NOT EXISTS SPCS_DB;
GRANT OWNERSHIP ON DATABASE SPCS_DB TO ROLE SPCS_ROLE COPY CURRENT GRANTS;

-- Step 4: warehouse
CREATE WAREHOUSE IF NOT EXISTS SPCS_WH WITH WAREHOUSE_SIZE = 'X-SMALL' AUTO_SUSPEND = 60;
GRANT USAGE ON WAREHOUSE SPCS_WH TO ROLE SPCS_ROLE;

-- Step 5: THE ONE PEOPLE FORGET — needed before any endpoint with public: true
GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE SPCS_ROLE;

-- Step 7: let yourself use the role
GRANT ROLE SPCS_ROLE TO USER IDENTIFIER(CURRENT_USER());

In [ ]:
%%sql
-- Step 6: the compute pool. An account-level object, and the thing you are billed for.
CREATE COMPUTE POOL IF NOT EXISTS SPCS_POOL
    MIN_NODES         = 1         -- required
    MAX_NODES         = 2         -- required
    INSTANCE_FAMILY   = CPU_X64_XS-- required; check SHOW COMPUTE POOL INSTANCE FAMILIES
    AUTO_RESUME       = TRUE      -- come back when a service is submitted
    AUTO_SUSPEND_SECS = 600       -- pool-level: fires when no work is scheduled on it.
                                  -- Default is 3600. The SERVICE has its own separate
                                  -- AUTO_SUSPEND_SECS, which defaults to 0 (off).
    INITIALLY_SUSPENDED = TRUE    -- do not start billing the moment you create it
    COMMENT = 'Deep-dive pool';

GRANT USAGE, MONITOR ON COMPUTE POOL SPCS_POOL TO ROLE SPCS_ROLE;

In [ ]:
%%sql -r instance_families
-- What instance families does this account actually offer? Never guess the name.
SHOW COMPUTE POOL INSTANCE FAMILIES;

In [ ]:
%%sql -r pool_state
-- Pool state. Read target_nodes vs active_nodes to see autoscaling in flight.
SHOW COMPUTE POOLS LIKE 'SPCS_POOL';

> ### ⚠️ Common misconceptions
>
> **"A compute pool with no services on it is not costing anything."**
> A pool bills in `IDLE`, `ACTIVE`, `STOPPING` and `RESIZING`. `IDLE` means the nodes are up and nothing is scheduled on them — which is exactly the state a forgotten pool sits in, quietly accruing node-hours. Only `SUSPENDED` is free.
> → [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)
>
> **"A compute pool lives in a database, like a stage or a table."**
> It is an **account-level** object, the same as a warehouse. You grant `USAGE` and `MONITOR` on it to a role; you do not qualify it with a database and schema. This shows up as a distractor in "which of these is schema-scoped" questions, alongside the image repository and the service, which genuinely are schema-scoped.
> → [SPCS overview](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)
>
> **"If I need more memory for a Python model, I need SPCS."**
> Not necessarily. A Snowpark-optimized warehouse gives you up to 1 TB of memory on a single node through `RESOURCE_CONSTRAINT`, with no Dockerfile and no pool to manage. What it cannot give you is a GPU or your own container image — and those, not memory, are the reasons to take on SPCS.
> → [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

In [ ]:
%%sql
-- Step 8: the image repository, created AS THE WORKLOAD ROLE so it owns it
USE ROLE SPCS_ROLE;
USE DATABASE SPCS_DB;
USE WAREHOUSE SPCS_WH;
CREATE SCHEMA IF NOT EXISTS APP;
CREATE IMAGE REPOSITORY IF NOT EXISTS SPCS_DB.APP.IMAGES;
CREATE STAGE IF NOT EXISTS SPCS_DB.APP.SPECS DIRECTORY = (ENABLE = TRUE);

In [ ]:
%%sql -r repo_url
-- You need this URL before you can tag anything. Copy the repository_url column.
SHOW IMAGE REPOSITORIES IN SCHEMA SPCS_DB.APP;

---
## 3. Step 9 — build and push, in a terminal

This half is not SQL, and it is where the platform mismatch bites.

```bash
# authenticate Docker against the Snowflake registry
snow spcs image-registry login

# BUILD FOR linux/amd64. Mandatory, and the quietest failure on Apple Silicon.
docker build --rm --platform linux/amd64 \
  -t <org>-<account>.registry.snowflakecomputing.com/spcs_db/app/images/my_api:latest .

docker push <org>-<account>.registry.snowflakecomputing.com/spcs_db/app/images/my_api:latest
```

The registry URL is `<orgname>-<acctname>.registry.snowflakecomputing.com/<database>/<schema>/<repository>`, lowercase. Take it from `SHOW IMAGE REPOSITORIES`, not from memory — a wrong hostname is rejected at push time, which is the good case.

> **In the specification you use the short path**, not the hostname:
> `image: /spcs_db/app/images/my_api:latest`

---
## 4. The specification

Two top-level keys: **`spec`** and **`serviceRoles`**. Everything else nests under `spec`.

```yaml
spec:
  containers:                 # at least one
    - name: api               # lowercase alphanumeric and '-', <= 63 chars,
                              # starts with a letter, ends alphanumeric
      image: /spcs_db/app/images/my_api:latest
      command: ["python"]     # overrides the image ENTRYPOINT
      args: ["server.py"]     # overrides CMD
      env:
        SERVER_PORT: 8000
      readinessProbe:         # Snowflake calls this to decide when you can serve traffic
        port: 8000
        path: /healthcheck
      resources:
        requests:             # what is guaranteed
          memory: 2Gi
          cpu: 0.5
          nvidia.com/gpu: 1   # a GPU is requested HERE
        limits:               # the ceiling
          memory: 4Gi
          cpu: 1
          nvidia.com/gpu: 1   # and must also appear as a limit, same quantity
      volumeMounts:
        - name: models
          mountPath: /app/models
      secrets:
        - snowflakeSecret: { objectName: my_secret }
          envVarName: API_KEY      # or directoryPath, for a file
          secretKeyRef: secret_string

  endpoints:
    - name: predict
      port: 8000              # or portRange, which requires TCP and non-public
      public: false           # true requires BIND SERVICE ENDPOINT ON ACCOUNT
      protocol: HTTP          # HTTP (default) or TCP

  volumes:
    - name: models
      source: stage           # local | memory | block | stage
      stageConfig: { name: "@spcs_db.app.model_stage" }

  logExporters:
    eventTableConfig: { logLevel: INFO }     # INFO (default) | ERROR | NONE

serviceRoles:
  - name: predictor
    endpoints: [predict]
```

Only `name` and `image` are required under a container. Everything else is optional and most of it you will want anyway.

### The fields that decide whether it works

| Field | What actually goes wrong |
|---|---|
| `resources.requests` / `limits` | A GPU must be requested **and** given a matching limit, and the pool must use a GPU instance family. Requesting more memory than any node in the pool has leaves the container unable to be scheduled |
| `endpoints[].public` | `true` without `BIND SERVICE ENDPOINT ON ACCOUNT` fails at `CREATE SERVICE`, not at first request |
| `readinessProbe` | Without one, traffic can reach a container that has not finished starting. Snowflake continuously calls the probe to decide when you are ready |
| `serviceRoles` | The mechanism for letting *another* role reach one endpoint but not the others — see section 7 |
| container `name` | Lowercase alphanumeric and `-`, starts with a letter, ends alphanumeric, at most 63 characters |
| platform | `linux/amd64`, always |

`logExporters.eventTableConfig.logLevel` defaults to `INFO`, which exports all user logs; `ERROR` exports stderr only; `NONE` turns off export to the event table. Setting `NONE` saves ingestion cost and removes the only record you will have when a container fails at 3am.

→ [More on the specification reference](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/specification-reference)

---
## 5. Step 10 — `CREATE SERVICE`, and the job alternative

```sql
CREATE SERVICE [ IF NOT EXISTS ] <name>
  IN COMPUTE POOL <pool>
  FROM SPECIFICATION $$ … $$                      -- inline
  -- or FROM @<stage> SPECIFICATION_FILE = 'spec.yaml'
  -- or FROM @<stage> SPECIFICATION_TEMPLATE_FILE = 'spec.tmpl' USING (KEY => 'value')
  [ AUTO_SUSPEND_SECS = <num> ]
  [ EXTERNAL_ACCESS_INTEGRATIONS = ( <eai> [ , … ] ) ]
  [ AUTO_RESUME = { TRUE | FALSE } ]              -- default TRUE
  [ MIN_INSTANCES = <num> ]                       -- default 1
  [ MIN_READY_INSTANCES = <num> ]
  [ MAX_INSTANCES = <num> ]                       -- defaults to MIN_INSTANCES
  [ LOG_LEVEL = '<log_level>' ]
  [ QUERY_WAREHOUSE = <warehouse> ]               -- warehouse the container uses for SQL back into Snowflake
  [ COMMENT = '…' ];
```

Two of these are worth reading twice.

- **`EXTERNAL_ACCESS_INTEGRATIONS`** is required for *any* outbound network call. A container with no integration cannot reach PyPI, your vendor's API, or anything else outside Snowflake. This is a default-deny, and it is the point.
- **`AUTO_SUSPEND_SECS`** on a service defaults to **0**, meaning auto-suspension is off. To enable it you must set 300 seconds or more. A service counts as idle when no queries are invoking its service functions and its status is `RUNNING`. Creating the service and never setting this is the standard way to end up paying for an endpoint nobody calls.

Required privileges: `CREATE SERVICE` on the schema, `USAGE` on the compute pool, `READ` on the specification stage and the image repository, and `BIND SERVICE ENDPOINT` on the account if any endpoint is public.

### Service or job service

| | `CREATE SERVICE` | `EXECUTE JOB SERVICE` |
|---|---|---|
| Lifetime | long-running; persists until suspended or dropped, and Snowflake restarts containers that exit | runs once and exits, like a stored procedure; containers are not restarted |
| Use for | inference endpoints, APIs, UIs | batch scoring, training runs, ETL |
| Blocking | no | synchronous by default; `ASYNC = TRUE` to detach, then `SPCS_WAIT_FOR()` to watch it |
| Parallelism | `MIN_INSTANCES` / `MAX_INSTANCES` | `REPLICAS = <n>`, with `SNOWFLAKE_JOBS_COUNT` and `SNOWFLAKE_JOB_INDEX` injected as environment variables so each replica takes a shard |
| Callable from SQL | yes, through a service function | **no — service functions cannot talk to a job service** |
| Suspended pool | suspended with the pool | keeps running until `DONE` or `FAILED`, then the nodes are released |

The decision is really about who waits. A long-running service means the *pool* waits, at cost, so the user does not. A job service means the user waits for a cold start, and you pay only for the run. "Nightly batch scoring" is a job service; "low-latency endpoint behind a dashboard" is a long-running service.

→ [More on working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

In [ ]:
%%sql
-- Long-running inference service
CREATE SERVICE IF NOT EXISTS SPCS_DB.APP.INFERENCE_SVC
  IN COMPUTE POOL SPCS_POOL
  FROM SPECIFICATION $$
spec:
  containers:
    - name: api
      image: /spcs_db/app/images/my_api:latest
      env:
        SERVER_PORT: 8000
      readinessProbe:
        port: 8000
        path: /healthcheck
      resources:
        requests: { memory: 2Gi, cpu: 0.5 }
        limits:   { memory: 4Gi, cpu: 1 }
  endpoints:
    - name: predict
      port: 8000
      public: false
  logExporters:
    eventTableConfig:
      logLevel: INFO
serviceRoles:
  - name: caller
    endpoints: [predict]
  $$
  MIN_INSTANCES = 1
  MAX_INSTANCES = 2
  QUERY_WAREHOUSE = SPCS_WH
  COMMENT = 'Model inference endpoint';

In [ ]:
%%sql
-- Batch alternative: runs once, exits, and is NOT stopped by suspending the pool.
-- REPLICAS injects SNOWFLAKE_JOBS_COUNT and SNOWFLAKE_JOB_INDEX so each replica takes a shard.
EXECUTE JOB SERVICE
  IN COMPUTE POOL SPCS_POOL
  NAME = SPCS_DB.APP.NIGHTLY_SCORING
  REPLICAS = 4
  ASYNC = TRUE
  FROM SPECIFICATION $$
spec:
  containers:
    - name: scorer
      image: /spcs_db/app/images/batch_scorer:latest
      env:
        TARGET_TABLE: SPCS_DB.APP.SCORES
  $$;

> ### 🤔 Stop and think
>
> - A long-running service gives you millisecond responses and bills for every hour it exists. A job service costs nothing between runs and makes the caller wait for a cold start. Where is the crossover for *your* traffic pattern — and how would you measure it rather than guess?
> - Suspending the pool when the team goes home saves real money and means the first query each morning is slow. Who gets to make that call: the team paying the bill, or the team answering for the latency?
> - You could build this as an SPCS service, or log the model to the registry and let Snowflake build the container for you. What do you lose by giving up the Dockerfile, and when is that loss worth it?

---
## 6. Monitoring — four commands, in the order you will actually need them

In [ ]:
%%sql -r svc_describe
-- Status and properties of the service
DESCRIBE SERVICE SPCS_DB.APP.INFERENCE_SVC;

In [ ]:
%%sql -r svc_containers
-- Per-container status: is it PENDING, READY, FAILED?
SHOW SERVICE CONTAINERS IN SERVICE SPCS_DB.APP.INFERENCE_SVC;

In [ ]:
%%sql -r svc_endpoints
-- The endpoint URL (ingress_url) once the endpoint is provisioned
SHOW ENDPOINTS IN SERVICE SPCS_DB.APP.INFERENCE_SVC;

In [ ]:
%%sql -r svc_logs
-- Container logs. instance index, container name, number of lines.
SELECT SYSTEM$GET_SERVICE_LOGS('SPCS_DB.APP.INFERENCE_SVC', 0, 'api', 100) AS logs;

### Reading what you get back

`DESCRIBE SERVICE` gives you the service's own status and properties. `SHOW SERVICE CONTAINERS IN SERVICE` drops a level, to the individual containers — this is where a service that looks created but is not working reveals itself. `SHOW ENDPOINTS IN SERVICE` gives the `ingress_url`, which does not appear until the endpoint is provisioned. `SYSTEM$GET_SERVICE_LOGS` is the last resort and the most informative.

A job service reports `PENDING`, `RUNNING`, `DONE`, `FAILED`, `CANCELLING` or `CANCELLED`; with `REPLICAS`, the job is `FAILED` if any instance fails, `DONE` only when all succeed. Service instances move through states such as `PENDING`, `READY` and `TERMINATING` — you can watch each instance transition during a rolling upgrade.

**A debug order that works:**

1. `SHOW SERVICE CONTAINERS` — is anything actually running?
2. If nothing is starting, check the pool (`SHOW COMPUTE POOLS`) for capacity and re-read the image path in the specification. A `resources.requests` larger than any node in the pool can supply will never schedule.
3. If a container started and stopped, `SYSTEM$GET_SERVICE_LOGS`. Its arguments are `(service_name, instance_id, container_name)` — instance ids start at 0, and the container name is the one from your specification — plus an optional line count and an optional flag to read logs from a **previously terminated** container, which is what you want after a crash loop.

→ [More on SYSTEM$GET_SERVICE_LOGS](https://docs.snowflake.com/en/sql-reference/functions/system_get_service_logs)

---
## 7. Reaching the service from SQL

```sql
CREATE FUNCTION SPCS_DB.APP.PREDICT(input VARCHAR)
  RETURNS VARCHAR
  SERVICE  = SPCS_DB.APP.INFERENCE_SVC
  ENDPOINT = 'predict'
  AS '/predict';                 -- the HTTP path your server serves
```

Three things must line up, and a mismatch in any of them produces an error that does not name which:

1. `SERVICE` — the service object.
2. `ENDPOINT` — the **name** from the specification's `endpoints[].name`. Not the port.
3. `AS '/path'` — the route your web server actually handles.

### What your container receives

Not one row. Snowflake sends **batches**, in the external-function format:

```
request   {"data": [[0, "first row value"], [1, "second row value"], …]}
response  {"data": [[0, "first result"],    [1, "second result"],   …]}
```

Your handler must iterate and echo each `row_index` back. Batching is tunable on the function with `MAX_BATCH_ROWS` (smaller batches allow more parallelism), `MAX_BATCH_RETRIES`, `ON_BATCH_FAILURE` and `BATCH_TIMEOUT_SECS`.

And the constraint that catches people designing backwards: **service functions cannot be used to communicate with a job service.** If SQL has to call it, it is a long-running service.

### Letting another role call it

Service roles are their own grant type:

```sql
GRANT SERVICE ROLE SPCS_DB.APP.INFERENCE_SVC!caller TO ROLE ANALYST_ROLE;
GRANT USAGE ON DATABASE SPCS_DB   TO ROLE ANALYST_ROLE;
GRANT USAGE ON SCHEMA SPCS_DB.APP TO ROLE ANALYST_ROLE;
```

`caller` there is a role you declared under `serviceRoles` in the specification, listing exactly which endpoints it reaches. There is also a built-in `<service>!all_endpoints_usage` for the case where you do not want that granularity — which is the trade: one grant, every endpoint.

### Object privileges on a service

| Privilege | Lets a role |
|---|---|
| `USAGE` | list the service with `SHOW` / `DESCRIBE` |
| `MONITOR` | inspect logs and container status |
| `OPERATE` | suspend, resume, upgrade a service; cancel a job service |
| `OWNERSHIP` | all of the above, plus modify properties and inspect the specification |

`MONITOR` without `OPERATE` is the useful combination for an on-call engineer who should be able to diagnose but not restart.

→ [More on service access control](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

In [ ]:
%%sql
-- Service function: the SQL door into the container
CREATE OR REPLACE FUNCTION SPCS_DB.APP.PREDICT(input VARCHAR)
  RETURNS VARCHAR
  SERVICE  = SPCS_DB.APP.INFERENCE_SVC
  ENDPOINT = 'predict'
  AS '/predict';

-- Let an analyst role reach the endpoint
GRANT SERVICE ROLE SPCS_DB.APP.INFERENCE_SVC!caller TO ROLE GENAI_ANALYST;
GRANT USAGE ON DATABASE SPCS_DB   TO ROLE GENAI_ANALYST;
GRANT USAGE ON SCHEMA SPCS_DB.APP TO ROLE GENAI_ANALYST;

In [ ]:
%%sql -r svc_function_call
-- Call it like any other function
SELECT ticket_id, SPCS_DB.APP.PREDICT(ticket_text) AS prediction
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 5;

> ### ⚠️ Common misconceptions
>
> **"I'll put the batch scoring job behind a service function so SQL can trigger it."**
> Service functions cannot communicate with a job service. They bind to an endpoint on a long-running service; a job service has no endpoint to bind to and exits when its work is done. Batch work is started with `EXECUTE JOB SERVICE`, not called from a `SELECT`.
> → [Service functions](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)
>
> **"My container gets one row per call, like a scalar UDF."**
> The request is a **batch**. Requests and responses both use the external-function shape: `{"data": [[row_index, col1, col2, …], …]}` going in, and `{"data": [[row_index, output], …]}` coming back. Your handler must loop over the rows and echo each `row_index` — a server written for one row at a time will return a single result for a whole batch and the mapping will be silently wrong.
> → [Service functions](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)
>
> **"Suspending the compute pool will kill my running batch job."**
> Suspending a pool suspends all services **except** job services. Running jobs continue until they reach `DONE` or `FAILED`, and only then are the nodes released. That is deliberate — it is what makes "suspend the pool at 6pm" a safe habit rather than a way to lose a half-finished run.
> → [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

---
## 8. Cost

### What you are billed for

**The compute pool, per node, by instance family.** Not the service, not the number of requests.

| Pool state | Billed? |
|---|---|
| `IDLE` — nodes up, nothing scheduled | **yes** |
| `ACTIVE` — at least one service running | **yes** |
| `RESIZING` | **yes** |
| `STOPPING` | **yes** |
| `SUSPENDED` — no running nodes | no |

`IDLE` is the one that surprises people. A pool with zero services still costs money until something suspends it.

### The two auto-suspends, which are not the same thing

There are two `AUTO_SUSPEND_SECS` settings and they operate at different levels:

- **On the compute pool**, default **3600**. It fires when the pool has no work scheduled on it.
- **On the service**, default **0**, meaning off; set 300 or more to enable. A service is idle when no queries are invoking its service functions and its status is `RUNNING`.

Leave the service one at its default and a deployed endpoint occupies the pool indefinitely, so the pool never reaches the state its own auto-suspend is waiting for, and you pay around the clock for a service nobody is calling. Setting the service-level value is what lets the pool go quiet on its own.

The manual equivalent stays useful for scheduled off-hours:

```sql
ALTER SERVICE SPCS_DB.APP.INFERENCE_SVC SUSPEND;    -- free the pool
ALTER COMPUTE POOL SPCS_POOL SUSPEND;               -- or suspend the pool directly
```

With `AUTO_RESUME = TRUE` on the pool, the next submitted service brings it back — at the cost of a cold start for whoever asks first. Wrapping this in a scheduled `TASK` is the common pattern, and it is a real trade: money for morning latency.

### Where the credits show up

| View | Use |
|---|---|
| `SNOWPARK_CONTAINER_SERVICES_HISTORY` | hourly credit usage, SPCS only |
| `METERING_HISTORY` where `SERVICE_TYPE = 'SNOWPARK_CONTAINER_SERVICES'` | hourly, alongside every other service type |
| `METERING_DAILY_HISTORY`, same filter | the daily roll-up |
| `DATA_TRANSFER_HISTORY` | outbound data transfer, which SPCS can generate and warehouses mostly do not |

Note the contrast with Cortex AI functions, which appear under `SERVICE_TYPE = 'AI_SERVICES'`. Two different values in the same column, and they are swapped in distractors constantly.

→ [More on SPCS cost views](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/accounts-orgs-usage-views)

In [ ]:
%%sql -r spcs_history
-- SPCS credits, hourly, SPCS only
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWPARK_CONTAINER_SERVICES_HISTORY
ORDER BY 1 DESC
LIMIT 50;

In [ ]:
%%sql -r spcs_metering
-- The same spend seen through METERING_HISTORY — note the SERVICE_TYPE value
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    NAME                          AS compute_pool,
    SUM(CREDITS_USED)             AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE SERVICE_TYPE = 'SNOWPARK_CONTAINER_SERVICES'
  AND START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY credits DESC;

In [ ]:
%%sql
-- Cost control: suspend the service so the pool can go idle, then suspend the pool
ALTER SERVICE SPCS_DB.APP.INFERENCE_SVC SUSPEND;
ALTER COMPUTE POOL SPCS_POOL SUSPEND;

-- Wake on demand (AUTO_RESUME = TRUE makes the pool come back by itself)
-- ALTER COMPUTE POOL SPCS_POOL RESUME;
-- ALTER SERVICE SPCS_DB.APP.INFERENCE_SVC RESUME;

---
## 9. Teardown — setup in reverse

```
1. ALTER SERVICE … SUSPEND      (or go straight to DROP SERVICE)
2. DROP SERVICE                 -- every service must be gone...
3. DROP COMPUTE POOL            -- ...or this is refused
4. DROP IMAGE REPOSITORY
5. DROP SCHEMA / DATABASE
```

You must stop all running services before you can drop a compute pool. Getting this wrong is not destructive — the drop simply fails — but it is a frequent exam item precisely because it is the only ordering constraint in teardown that the system enforces for you.

---

## Check your understanding

Twelve questions on this deep dive, weighted toward design judgement. Answer before expanding.

**1.** Which compute pool states are billed?

<details><summary>Show answer</summary>

`IDLE`, `ACTIVE`, `STOPPING` and `RESIZING`. `SUSPENDED` is the only state with no running nodes and therefore no charge. `IDLE` is the one that matters operationally: nodes are up with nothing scheduled on them, which is exactly where an abandoned pool sits.

→ [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

</details>

**2.** Which three parameters are required by `CREATE COMPUTE POOL`?

<details><summary>Show answer</summary>

`MIN_NODES`, `MAX_NODES` and `INSTANCE_FAMILY`. Everything else — `AUTO_RESUME`, `AUTO_SUSPEND_SECS`, `INITIALLY_SUSPENDED`, `COMMENT` — has a default. Do not guess an instance family name: run `SHOW COMPUTE POOL INSTANCE FAMILIES` and read what this account actually offers, since GPU families in particular vary by region and cloud.

→ [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

</details>

**3.** What is the default `AUTO_SUSPEND_SECS` on a compute pool, and on a service?

<details><summary>Show answer</summary>

**3600 seconds on the pool**, and **0 on the service** — 0 meaning auto-suspension is disabled, with 300 seconds the minimum you can set to enable it. The asymmetry is the whole cost story of SPCS: the pool's default will eventually save you, but only once nothing is occupying it, and a service left at its own default never stops occupying it.

→ [CREATE SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-service)

</details>

**4.** `CREATE SERVICE` succeeds, but nothing ever becomes ready and the logs show the container was never pulled. Where do you look first?

<details><summary>Show answer</summary>

`SHOW SERVICE CONTAINERS IN SERVICE`, then the two usual causes: the image was never pushed (or the path in the specification does not match the repository), or `resources.requests` asks for more memory or CPU than any node in the pool can provide, so the container can never be scheduled. Both produce "created but not running" rather than an error from `CREATE SERVICE` — the statement only validates the specification, not the world it describes.

→ [Working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>

**5.** A team builds their image on an Apple Silicon laptop. `docker push` succeeds and the containers crash-loop. What happened?

<details><summary>Show answer</summary>

The image was built for `arm64`. SPCS requires `linux/amd64`, so the fix is `docker build --platform linux/amd64`. The reason this wastes an afternoon is that every step before the crash succeeds: the build works, the tag is valid, the push is accepted, the service is created. Nothing tells you the architecture is wrong until containers start and immediately die.

→ [SPCS overview](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)

</details>

**6.** A developer wants nightly batch scoring triggered from a SQL task, and plans to expose the job through a service function. What is wrong, and what should they do?

<details><summary>Show answer</summary>

Service functions cannot communicate with a job service. They bind to an endpoint on a long-running service, and a job service has no endpoint and exits when done. The batch path is `EXECUTE JOB SERVICE`, called directly from the task — optionally with `ASYNC = TRUE` and `SPCS_WAIT_FOR()` if the task should not block. The tempting alternative — keep a long-running service up so SQL *can* call it — works, and costs you node-hours all day for a job that runs for twenty minutes at 2am.

→ [Service functions](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>

**7.** A container returns one prediction for a query over 10,000 rows, and every row gets the same answer. What is wrong with the server?

<details><summary>Show answer</summary>

It was written for a single row. A service function sends a **batch**: `{"data": [[row_index, col1, …], …]}`, and expects `{"data": [[row_index, output], …]}` back, with each `row_index` echoed. A handler that reads only the first element and returns one result produces a response Snowflake maps onto the batch — no error, just uniformly wrong output. `MAX_BATCH_ROWS` controls how large those batches get, but it cannot fix a handler that ignores the shape.

→ [Service functions](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>

**8.** An analyst role needs to call `PREDICT()`. What do you grant, and what is the alternative?

<details><summary>Show answer</summary>

`GRANT SERVICE ROLE <service>!<role_name>` for a service role you declared under `serviceRoles` in the specification, plus `USAGE` on the database and the schema holding the function. The declared service role names exactly which endpoints it reaches. The alternative is the built-in `<service>!all_endpoints_usage`, which is one grant instead of a specification edit and gives that role every endpoint on the service — including any admin or debug endpoint you add later without thinking about it.

→ [Working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>

**9.** Your team needs 200 GB of memory for a Python feature-engineering job. Compute pool or Snowpark-optimized warehouse?

<details><summary>Show answer</summary>

Warehouse, almost certainly. A Snowpark-optimized warehouse is single-node with memory set by `RESOURCE_CONSTRAINT` — `MEMORY_16X` gives 256 GB at minimum size M, and `MEMORY_64X` gives 1 TB at minimum size L. No image, no pool, no ordering to get right, and it suspends on the warehouse rules you already operate. Move to a compute pool when you need a **GPU** or your **own container image**, because those are the two things a warehouse cannot do at any size.

→ [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool)

</details>

**10.** An inference endpoint serves 30 requests a day, in bursts, during office hours. Design the cost profile and say what each option costs.

<details><summary>Show answer</summary>

Three honest options. **Leave it up:** every request is fast and you pay node-hours 24/7 for roughly 30 calls. **Set the service's `AUTO_SUSPEND_SECS`** to something like 600 — the endpoint goes quiet between bursts, the pool can then reach its own idle timeout, and whoever sends the first request after a gap pays a cold start. **Schedule it** with a task that suspends at 18:00 and resumes at 08:00 — predictable cost and predictable latency, but useless for anyone working late. There is no option where you get instant responses and no idle cost; you are choosing which one to give up.

→ [SPCS cost views](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/accounts-orgs-usage-views)

</details>

**11.** You are reviewing a specification that sets `logExporters.eventTableConfig.logLevel: NONE` and omits `readinessProbe`. What do you say in review?

<details><summary>Show answer</summary>

Both are defensible individually and dangerous together. `NONE` turns off export to the event table, saving ingestion cost and removing the record you will want when something fails unattended; `ERROR` keeps stderr for far less than `INFO`. Omitting `readinessProbe` means Snowflake has no signal for when the container can serve, so traffic can reach a process that is still starting — intermittent failures under load, which is the hardest class to reproduce. With logging off as well, you will be diagnosing that from nothing.

→ [Specification reference](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/specification-reference)

</details>

**12.** *(connects to Domain 3 — governance and cost)* Finance asks why the AI bill doubled. How do you separate Cortex AI function spend from SPCS spend, and what does each number actually represent?

<details><summary>Show answer</summary>

They are different `SERVICE_TYPE` values in the same views: `'AI_SERVICES'` for Cortex AI functions and `'SNOWPARK_CONTAINER_SERVICES'` for SPCS, in `METERING_HISTORY` (hourly) and `METERING_DAILY_HISTORY` (daily). `SNOWPARK_CONTAINER_SERVICES_HISTORY` gives SPCS alone at hourly grain. What the numbers mean differs in kind: AI-function spend scales with *calls and tokens*, so it falls when usage falls; SPCS spend is *node-hours*, so it does not fall when usage falls — only when something suspends. A doubled bill with flat usage points at SPCS, and specifically at a pool nobody suspended.

→ [SPCS cost views](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/accounts-orgs-usage-views)

</details>